# Relaxation Stability (Budgeted Strategy-Level Comparison)

This notebook implements the correct strategy-level comparison:

- Pool size: N (default 24)
- Budget: K (default 8)
- Proxy baseline: median steps of proxy_topK
- Random baseline: repeated resampling of K from full pool (S trials)

The boxplot shows the distribution of random median steps under fixed budget K,
with proxy median plotted as a horizontal reference line.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================
# Parameters (edit if needed)
# ==============================
csv_path = 'relaxation_performance_stats.csv'
K = 8
S = 2000  # number of resampling trials
random_seed = 0

rng = np.random.default_rng(random_seed)

# ==============================
# Load data
# ==============================
df = pd.read_csv(csv_path)

# Identify proxy group automatically
proxy_group = [g for g in df['group'].unique() if 'proxy' in g][0]

df_proxy = df[df['group'] == proxy_group].copy()
df_pool = df.copy()

assert len(df_pool) >= K, 'Pool size must be >= K'

# ==============================
# Proxy baseline (strategy-level)
# ==============================
proxy_median_steps = np.mean(df_proxy['relax_steps'].values)

# ==============================
# Random strategy resampling
# ==============================
pool_steps = df_pool['relax_steps'].values
N = len(pool_steps)

random_medians = np.empty(S)
for i in range(S):
    idx = rng.choice(N, size=K, replace=False)
    random_medians[i] = np.mean(pool_steps[idx])

# ==============================
# Plot
# ==============================
plt.figure(figsize=(5, 4), dpi=150)

plt.boxplot(random_medians, widths=0.4)
plt.axhline(proxy_median_steps, linestyle='--')

plt.ylabel('Median relaxation steps (budget K)')
plt.xticks([1], ['Random (resampled)'])

plt.title('Optimization Stability Under Fixed Budget')
plt.tight_layout()
plt.show()


IndexError: list index out of range